# Notebook 04: Carga en Base de Datos
## Proyecto II Parcial — Modelado Avanzado de Base de Datos - 30759
## Integrantes:
- Naomi Obando
- Mauri Tandazo

**Fase 7:** Carga idempotente de todas las tablas en DuckDB + verificación SQL obligatoria.

---
### ¿Por qué DuckDB?
- Base de datos **columnar analítica** embebida (sin servidor)
- Compatible con SQL estándar y con Pandas/Parquet nativamente
- Rendimiento OLAP excepcional para datasets medianos
- Sin dependencias externas, ideal para entornos locales y académicos
- Idempotencia garantizada mediante `process_id` por ejecución

### Tablas cargadas
| Tabla | Descripción |
|---|---|
| `gold_trips_clean` | Viajes limpios y validados (granular) |
| `gold_daily_revenue` | Ingresos diarios por servicio |
| `gold_location_performance` | Rendimiento por zona |
| `quality_rejected_records` | Registros rechazados con causa |
| `quality_metrics_summary` | Métricas de calidad por período |
| `audit_file_inventory` | Inventario técnico de archivos |

In [3]:
import sys, os, duckdb
sys.path.insert(0, os.path.abspath('..'))

from src.utils          import generate_process_id, load_config, setup_logger, create_spark_session
from src.schema_recovery import read_bronze_layer
from src.transformations  import run_transformation_phase, save_silver_layer, read_silver_layer
from src.quality_rules    import run_quality_phase
from src.load             import (
    get_duckdb_connection, initialize_schema,
    run_load_phase, run_verification_queries, query_duckdb
)
import json

PROCESS_ID  = generate_process_id()
CONFIG_PATH = '../config/etl_config.yaml'
config      = load_config(CONFIG_PATH)
config['paths']    = {k: f'../{v}' for k, v in config['paths'].items()}
config['paths']['metadata_dir'] = '../metadata'
config['database']['path'] = '../data/warehouse/nyc_tlc.duckdb'

logger = setup_logger('nb04_carga', '../logs', PROCESS_ID)
spark  = create_spark_session(config)
print(f'process_id : {PROCESS_ID}')
print(f'DuckDB     : {config["database"]["path"]}')

process_id : ETL-20260623-155152-6457D0A0
DuckDB     : ../data/warehouse/nyc_tlc.duckdb


In [6]:
# ── 1. Leer capa Silver y construir tablas Gold ───────────────────
silver_df = read_silver_layer(spark, config['paths']['silver_dir'], logger)

if silver_df is None:
    print('⚠️ Silver no encontrada. Ejecuta primero los Notebooks 02 y 03.')
else:
    print(f'Silver cargada: {silver_df.count():,} registros')
    gold_tables = run_quality_phase(spark, silver_df, config, PROCESS_ID, logger)
    print(f'\nTablas Gold generadas: {list(gold_tables.keys())}')

2026-06-23 15:52:44 | INFO     | nb04_carga                     | Capa silver cargada: 44,488,872 registros
Silver cargada: 44,488,872 registros
2026-06-23 15:52:45 | INFO     | nb04_carga                     | ══════════════════════════════════════════════════════════════════════
2026-06-23 15:52:45 | INFO     | nb04_carga                     |   FASE 5-6: CALIDAD Y GOLD  —  process_id: ETL-20260623-155152-6457D0A0
2026-06-23 15:52:45 | INFO     | nb04_carga                     | ══════════════════════════════════════════════════════════════════════
2026-06-23 15:52:45 | INFO     | nb04_carga                     | [5.1] Aplicando reglas de calidad...
2026-06-23 15:52:45 | INFO     | nb04_carga                     |   Aplicando 11 reglas de calidad...
2026-06-23 15:52:57 | INFO     | nb04_carga                     |   Válidos    : 44,250,454
2026-06-23 15:53:07 | INFO     | nb04_carga                     |   Rechazados : 430,582
2026-06-23 15:53:07 | INFO     | nb04_carga              

In [7]:
# ── 2. Cargar inventario de archivos desde checkpoint ─────────────
import glob
inv_files = sorted(glob.glob('../data/audit/inventory_*.json'))
inventory_records = []
if inv_files:
    with open(inv_files[-1]) as f:
        inventory_records = json.load(f)
    print(f'Inventario cargado: {len(inventory_records)} archivos')
else:
    print('⚠️ Sin inventario previo. Ejecuta el Notebook 01 primero.')

Inventario cargado: 19 archivos


In [8]:
# ── 3. Ejecutar carga completa en DuckDB ──────────────────────────
verification_results = run_load_phase(
    spark, gold_tables, inventory_records, config, PROCESS_ID, logger
)
print(f'\nConsultas de verificación ejecutadas: {len(verification_results)}')
print(f'Exitosas: {sum(1 for r in verification_results if r["status"]=="OK")}')

2026-06-23 16:06:09 | INFO     | nb04_carga                     | ══════════════════════════════════════════════════════════════════════
2026-06-23 16:06:09 | INFO     | nb04_carga                     |   FASE 7: CARGA EN DUCKDB (Parquet nativo)  —  ETL-20260623-155152-6457D0A0
2026-06-23 16:06:09 | INFO     | nb04_carga                     |   Motor: DuckDB — ../data/warehouse/nyc_tlc.duckdb
2026-06-23 16:06:09 | INFO     | nb04_carga                     | ══════════════════════════════════════════════════════════════════════
2026-06-23 16:06:09 | INFO     | nb04_carga                     | 
[7.1] Inicializando esquema DuckDB...
2026-06-23 16:06:09 | INFO     | nb04_carga                     |   Inicializando esquema DuckDB...
2026-06-23 16:06:09 | INFO     | nb04_carga                     |     [OK] Tabla lista: gold_trips_clean
2026-06-23 16:06:09 | INFO     | nb04_carga                     |     [OK] Tabla lista: gold_daily_revenue
2026-06-23 16:06:09 | INFO     | nb04_carga       

In [9]:
# ── 4. VERIFICACIÓN SQL OBLIGATORIA 1 — Viajes por servicio ───────
import pandas as pd
pd.set_option('display.float_format', '{:,.2f}'.format)

print('╔══════════════════════════════════════════════════════════════╗')
print('║  VERIFICACIÓN 1: Total de viajes e ingresos por servicio     ║')
print('╚══════════════════════════════════════════════════════════════╝')

result = query_duckdb(config, """
SELECT
    service_type,
    COUNT(*)                         AS total_trips,
    ROUND(SUM(total_amount), 2)      AS total_revenue,
    ROUND(AVG(fare_amount), 2)       AS avg_fare,
    ROUND(AVG(trip_duration_minutes),2) AS avg_duration_min,
    ROUND(AVG(trip_distance), 2)     AS avg_distance_miles
FROM gold_trips_clean
GROUP BY service_type
ORDER BY total_revenue DESC;
""")
display(result)

╔══════════════════════════════════════════════════════════════╗
║  VERIFICACIÓN 1: Total de viajes e ingresos por servicio     ║
╚══════════════════════════════════════════════════════════════╝


,service_type,total_trips,total_revenue,avg_fare,avg_duration_min,avg_distance_miles
0,yellow,25653020,"599,270,700.33",15.89,14.21,4.09
1,fhvhv,18465479,"506,049,833.01",21.57,18.24,4.87
2,green,131955,"2,895,741.12",16.70,14.11,10.90


In [10]:
# ── 5. VERIFICACIÓN SQL OBLIGATORIA 2 — Métricas de calidad ───────
print('╔══════════════════════════════════════════════════════════════╗')
print('║  VERIFICACIÓN 2: Métricas de calidad por servicio/año/mes   ║')
print('╚══════════════════════════════════════════════════════════════╝')

result = query_duckdb(config, """
SELECT service_type, year, month,
       total_records, valid_records, rejected_records,
       suspicious_records,
       ROUND(quality_percentage, 2) AS quality_pct
FROM quality_metrics_summary
ORDER BY year, month, service_type;
""")
display(result)

╔══════════════════════════════════════════════════════════════╗
║  VERIFICACIÓN 2: Métricas de calidad por servicio/año/mes   ║
╚══════════════════════════════════════════════════════════════╝


,service_type,year,month,total_records,valid_records,rejected_records,suspicious_records,quality_pct
0,yellow,2019,12,409230,109,409121,3,0.03
1,yellow,2020,1,6762629,6353508,409121,71925,93.95
2,yellow,2020,2,409151,30,409121,1,0.01
3,yellow,2020,3,409126,5,409121,0,0.00
4,yellow,2020,4,409122,1,409121,0,0.00
5,yellow,2020,5,409126,5,409121,0,0.00
6,yellow,2020,6,409122,1,409121,0,0.00
7,yellow,2020,7,409127,6,409121,0,0.00
8,yellow,2020,12,409134,13,409121,0,0.00
9,yellow,2021,1,1762775,1353654,409121,22218,76.79


In [11]:
# ── 6. VERIFICACIÓN SQL OBLIGATORIA 3 — Top 20 rutas ─────────────
print('╔══════════════════════════════════════════════════════════════╗')
print('║  VERIFICACIÓN 3: Top 20 rutas por ingresos totales          ║')
print('╚══════════════════════════════════════════════════════════════╝')

result = query_duckdb(config, """
SELECT
    pickup_location_id,
    dropoff_location_id,
    COUNT(*)                        AS total_trips,
    ROUND(SUM(total_amount), 2)     AS total_revenue,
    ROUND(AVG(trip_duration_minutes),2) AS avg_duration
FROM gold_trips_clean
GROUP BY pickup_location_id, dropoff_location_id
ORDER BY total_revenue DESC
LIMIT 20;
""")
display(result)

╔══════════════════════════════════════════════════════════════╗
║  VERIFICACIÓN 3: Top 20 rutas por ingresos totales          ║
╚══════════════════════════════════════════════════════════════╝


,pickup_location_id,dropoff_location_id,total_trips,total_revenue,avg_duration
0,132,265,99681,"11,324,494.97",46.16
1,138,265,44601,"5,042,507.53",43.38
2,132,230,47330,"4,068,230.68",51.48
3,264,264,148709,"3,778,062.95",13.56
4,138,230,49673,"3,292,665.14",34.88
5,132,48,32156,"2,684,704.04",51.64
6,237,236,194007,"2,645,730.27",6.92
7,230,138,38584,"2,510,251.70",32.03
8,236,237,167039,"2,384,820.64",8.09
9,132,164,26587,"2,261,705.72",42.97


In [12]:
# ── 7. Verificaciones adicionales ─────────────────────────────────
print('=== INVENTARIO DE ARCHIVOS POR ESTADO ===')
display(query_duckdb(config, """
SELECT service_type, read_status,
       COUNT(*) AS files, SUM(record_count) AS records
FROM audit_file_inventory
GROUP BY service_type, read_status
ORDER BY service_type;
"""))

print('\n=== VIAJES SOSPECHOSOS POR SERVICIO ===')
display(query_duckdb(config, """
SELECT service_type,
       COUNT(*) total,
       SUM(CASE WHEN is_suspicious_trip THEN 1 ELSE 0 END) suspicious,
       ROUND(SUM(CASE WHEN is_suspicious_trip THEN 1 ELSE 0 END)::DOUBLE
             / COUNT(*)::DOUBLE * 100, 2) AS pct_suspicious
FROM gold_trips_clean
GROUP BY service_type;
"""))

print(f'\n✅ Fase 7 completada — process_id: {PROCESS_ID}')
print(f'   Base de datos: {config["database"]["path"]}')
spark.stop()

=== INVENTARIO DE ARCHIVOS POR ESTADO ===


,service_type,read_status,files,records
0,bad_parquet,RECOVERABLE_SCHEMA_MISMATCH,2,0.00
1,bad_parquet,SUCCESS,5,"22,192.00"
2,bad_parquet,NOT_RECOVERABLE_UNSUPPORTED_FORMAT,1,0.00
3,fhvhv,SUCCESS,1,"18,479,031.00"
4,green,SUCCESS,2,"133,020.00"
5,yellow,SUCCESS,8,"25,890,876.00"



=== VIAJES SOSPECHOSOS POR SERVICIO ===


,service_type,total,suspicious,pct_suspicious
0,fhvhv,18465479,"12,504.00",0.07
1,green,131955,"6,315.00",4.79
2,yellow,25653020,"338,983.00",1.32



✅ Fase 7 completada — process_id: ETL-20260623-155152-6457D0A0
   Base de datos: ../data/warehouse/nyc_tlc.duckdb
